# Debugging and Reflection

### Fitness Tracker Project — Part 5

*Class 19*

📖 [Full reading: Class 19](https://BU-EK125.github.io/EK125/class/Class19.html)
📋 [Today's Team Activity](https://BU-EK125.github.io/EK125/gpps/Class19_TeamActivity.html)

## This Morning: The Bug Hunt

You'll get another team's Fitness Tracker code — same contract,
different code. Five bugs are hiding inside: one syntax, two
runtime, two logic. Debug it until your report matches theirs
exactly.

### Takeaway

This is Exam 2 rehearsal — the same three skills you'll use on
paper: classify a bug, read a traceback, and debug code you
didn't write.

## The Debugging Method

| Step | In code |
|---|---|
| **Reproduce** | Run it and get the error (or wrong answer) every time |
| **Locate** | Read the traceback bottom-up, or print at the seams |
| **Hypothesize** | "I think `best_day` is a position, not a day number" |
| **Change ONE thing** | Make one fix. Run again. |
| **Confirm** | Did the error go away *and* is the answer right? |

### Takeaway

One change, one run, every time. Change five things at once and
you won't know which one fixed it — or what else you broke.

## Three Kinds of Bugs

| Kind | What you see | How you catch it |
|---|---|---|
| **Syntax** | Program doesn't start at all | Read the line shown, and the one before it |
| **Runtime** | Program starts, then crashes with a traceback | Read bottom-up; match the error to its cause |
| **Logic** | Program runs fine, answer is wrong | Tests, reasonableness checks, known-good answers |

**Syntax — the program never starts:**
```python
for day in days
    print(day)
```
```python
if steps = 10000:
    print("Goal met")
```
Missing colon; and `=` where `==` belongs. Both stop Python
before a single line runs — these are grammar mistakes, not bugs
in your logic.

**Runtime — the program starts, then crashes:**
```python
print("Steps: " + steps)
```
```python
best = get_best_day(data)  # prints only
print(best + 1)
```
The first crashes on the spot: `str` plus `int`. The second
crashes far from its real cause — `get_best_day` is the problem,
not the `+ 1` line.

**Logic — no crash, just a wrong answer:**
```python
total = 0
for v in values:
    total = 0  # resets every pass!
    total = total + v
```
```python
if resting_hr < goal:  # should be <=
    count += 1
```
The first returns the last value, not a sum — `total = 0` belongs
once, before the loop. The second silently drops any value that
exactly equals the goal.

**A "logic" bug that crashes anyway:**
```python
for i in range(len(days)):
    if values[i + 1] < values[i]:
        drops += 1
```
The intent is a logic mistake — comparing the wrong pair of
values. But on the last pass, `i + 1` runs off the end of the
list: `IndexError`. Not every off-by-one stays silent.

**Scope — a variable that doesn't exist where you think:**
```python
def analyze(data):
    total_steps = sum(data["steps"])
    return total_steps

analyze(data)
print(total_steps)  # NameError
```
`total_steps` only exists inside `analyze`. Once the function
returns, it's gone — use the return value, not the variable's
name.

### Takeaway

Syntax errors stop the program before it starts. Runtime errors
stop it partway through. Logic errors don't stop it at all — so
you have to go looking for them.

## Reading a Multi-File Traceback

Read every traceback from the **bottom up**: last line is what
went wrong, the block above is where, the blocks above that are
how you got there. The crash site and the bug site are often two
different places.

**Same file, different line:**
```python
values = line.strip().split(",")
if values != "":          # always True
    steps = int(values[0])  # crashes here
```
```
ValueError: invalid literal for int()
```
The crash is on the `int(...)` line, but the real mistake is the
`if` two lines above — comparing a list to a string is always
`True`, so blank rows are never skipped.

**Crossing files — a contract mismatch:**
```
Traceback (most recent call last):
  File "main.py", line 15, in main
    write_report(results, "report.txt")
  File "reporter.py", line 9, in write_report
    f"Goal met: {r['days_meeting_goal']}"
KeyError: 'days_meeting_goal'
```
```python
# Last line of analyze_metric in analyzer.py:
return {"metric": metric, "met_goal": count}
```
The crash is in `reporter.py`, but the dictionary was built in
`analyzer.py` — it returns `met_goal`, not the contract's
`days_meeting_goal`. Whoever owns `analyzer.py` fixes it.

### Takeaway

The error appears where a bad value is **used**. The bug is
usually where that value was **created**. Follow it back across
the seam, and let the contract decide who's off-contract.

## Debugging Code You Didn't Write

1. **Read the contract** — what should it do?
2. **Read `main.py`** — how is it called, what happens to the result?
3. **Run it** — get the error or the wrong output
4. **Follow the data** — print at the seams, find the first value that's wrong
5. **Only now, read that function's code closely**

### Takeaway

The contract tells you what "right" means. Without it, you're
just guessing what the author intended.

## Writing Tests That Catch the Bug

A good test is aimed at a specific way the code could be wrong —
not just any input.

**The tie-breaking case:**
```python
tied = {
    "day": [1, 2, 3],
    "steps": [900, 950, 950],
}
check("earliest tie wins", 2, best_day(tied))
```
Two days tie for the lead. If `>` quietly became `>=`, this test
catches it — a list with no tie wouldn't.

**The boundary case:**
```python
check("goal met exactly", 1,
      count_meeting_goal([8000], 8000))
```
A value clearly above or below the goal passes whether the code
uses `>` or `>=`. Only a value exactly *at* the goal tells them
apart.

**The smallest possible input:**
```python
check("empty list", 0, average_steps([]))
```
A function that divides by `len(values)` crashes here with
`ZeroDivisionError` unless it's handled — and an empty list is a
completely legal input to hand it.

**The rounding case:**
```python
check("rounds to one place", 66.7,
      round(200.15 / 3, 1))
```
Pick a value where the *unrounded* result would fail an exact
match — so the test is actually checking the rounding, not just
getting lucky.

### Takeaway

Four questions worth asking about any function: what about a
tie? What about the exact boundary? What about empty input? What
about rounding?

## Fix It So It Stays Fixed

When you find a bug, ask: **"Why didn't my tests catch this?"**
Then add a **regression test** — one that would have caught it —
so if the bug ever comes back, the test fails immediately.

```python
# Regression test: exact goal must count
check("at goal", True, meets_goal(62, 62))
```

### Takeaway

You'll add exactly this kind of test to your own file this
afternoon, naming the bug it guards against.

## This Afternoon

- **Live debugging** of the two logic bugs together — trace
  tables, and an optional look at the PyCharm debugger
- **Add a regression test** to your own project file
- **Team retrospective**
- **Start your individual reflection** — finish it for homework

## The Retrospective

| | Question |
|---|---|
| **Continue** | What worked well that we should keep doing? |
| **Stop** | What caused problems that we should stop doing? |
| **Start** | What should we try next time that we didn't do this time? |

Two ground rules: **blame the process, not the person** — almost
every bug traces back to something the process allowed. And **be
specific** — "communication was bad" helps no one.

**Try rewriting this one:** *"Sam was slow and communication was
bad."*

A specific, process-focused version: *"We didn't agree in Class
15 on who checks the Drive folder before class, so Step 2 used
stale files."*

### Takeaway

Your integration log and Decision Log are your evidence for
this — use them instead of relying on memory.

## Due Tuesday

Gradescope, by **Tuesday 11/10, 11:59 PM**:

- **Team:** Bug Log (all five bugs) and the fixed files; the
  team retrospective
- **Individual:** your file with its new regression test; your
  ~300-word reflection, answering all five prompts with specific
  examples

## Quick Reference: Classes 9–19

Every class since Exam 1 shows up somewhere on Friday. If
something below feels rusty, the reading is one click away.

| Class | Where it shows up |
|---|---|
| [9: Moving to PyCharm](https://BU-EK125.github.io/EK125/class/Class9.html) | Imports, running a file, where a variable lives |
| [10: Mastering Slicing](https://BU-EK125.github.io/EK125/class/Class10.html) | Slicing with a step, list comprehensions |
| [11: NumPy Arrays](https://BU-EK125.github.io/EK125/class/Class11.html) | Array math vs. list math, shape, indexing |
| [12: Dictionaries](https://BU-EK125.github.io/EK125/class/Class12.html) | `.get()`, keys/values, a dictionary of lists |
| [13: User-Defined Functions](https://BU-EK125.github.io/EK125/class/Class13.html) | Return vs. print, scope, default parameters |

| Class | Where it shows up |
|---|---|
| [14: File I/O](https://BU-EK125.github.io/EK125/class/Class14.html) | `with`, reading/writing a file, skipping blank rows |
| [15: Project Kickoff](https://BU-EK125.github.io/EK125/class/Class15.html) | Contracts as a shared vocabulary (less exam weight) |
| [16: Stub Building](https://BU-EK125.github.io/EK125/class/Class16.html) | A faithful stub, reading a contract closely |
| [17: Writing Real Functions](https://BU-EK125.github.io/EK125/class/Class17.html) | `check()` tests, boundary values, position vs. label |
| [18: Integrating It All Together](https://BU-EK125.github.io/EK125/class/Class18.html) | Multi-file tracebacks, the right integration order |

## Exam 2: What's Fair Game

Exam 2 is **Friday 11/13**. Today's skills are directly tested:

- Label a bug as syntax, runtime, or logic
- Match an error message to its likely cause
- Fix a string-plus-int concatenation
- Find a `<` vs `<=` boundary bug against a docstring, and write
  the `check()` that catches it
- Find a position-vs-label bug
- Trace a loop and find the first line where it differs from
  what's expected